# 08 - Run capped full-cohort DNABERT2 embeddings

The four-isolate DNABERT2 MPS smoke test passed. This notebook creates a compact full-cohort window table with 16 evenly spaced windows per isolate, extracts frozen DNABERT2 embeddings through the prepared PyTorch-attention fallback snapshot, and evaluates the same lineage-held-out split used by the k-mer baseline.

## 1. Configuration

Execution switches default to `False`. The sampled window CSV should contain at most 3,152 rows and remain small. Embedding extraction uses batch size 1 on Apple MPS.

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_public_assemblies.csv"
SPLITS_PATH = PROJECT_ROOT / "data" / "processed" / "isolate_splits.csv"
MODEL_DIR = PROJECT_ROOT / "data" / "external" / "dnabert2_pytorch_fallback"
WINDOWS_PATH = PROJECT_ROOT / "data" / "processed" / "dnabert2_full_sampled_windows.csv"
EMBEDDINGS_PATH = PROJECT_ROOT / "data" / "processed" / "dnabert2_full_embeddings.npz"
METRICS_PATH = PROJECT_ROOT / "reports" / "dnabert2_full_embedding_baseline.json"
MAXIMUM_WINDOWS_PER_ISOLATE = 16
BATCH_SIZE = 1
RUN_SAMPLED_WINDOWS = False
RUN_FULL_EMBEDDINGS = False
RUN_FULL_EMBEDDING_BASELINE = False

print("Project root:", PROJECT_ROOT)

## 2. Review the full-cohort workload

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, dtype={"label": str}).fillna("")
print("Isolates:", len(manifest))
print("Maximum sampled windows:", len(manifest) * MAXIMUM_WINDOWS_PER_ISOLATE)
print("Prepared model exists:", MODEL_DIR.exists())
print("Free disk space: %.2f GB" % (shutil.disk_usage(PROJECT_ROOT).free / 1_000_000_000))
print(pd.crosstab(manifest["split"], manifest["label"], margins=True))

## 3. Build the compact sampled-window table

Change `RUN_SAMPLED_WINDOWS` to `True` after reviewing the workload.

In [ ]:
windows_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "build_sampled_windows.py"),
    "--manifest",
    str(MANIFEST_PATH),
    "--output",
    str(WINDOWS_PATH),
    "--maximum-windows-per-isolate",
    str(MAXIMUM_WINDOWS_PER_ISOLATE),
]
print(" ".join(windows_command))
if RUN_SAMPLED_WINDOWS:
    subprocess.run(windows_command, cwd=PROJECT_ROOT, check=True)
if WINDOWS_PATH.exists():
    windows = pd.read_csv(WINDOWS_PATH)
    print("Sampled windows:", len(windows))
    print("Isolates:", windows["isolate_id"].nunique())
    print("CSV size: %.2f MB" % (WINDOWS_PATH.stat().st_size / 1_000_000))

## 4. Extract full-cohort frozen embeddings

Change `RUN_FULL_EMBEDDINGS` to `True` after the sampled windows exist. Keep batch size 1 for the first full MPS run.

In [ ]:
embedding_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "embed_dnabert2.py"),
    "--model",
    str(MODEL_DIR),
    "--windows",
    str(WINDOWS_PATH),
    "--output",
    str(EMBEDDINGS_PATH),
    "--maximum-windows-per-isolate",
    str(MAXIMUM_WINDOWS_PER_ISOLATE),
    "--batch-size",
    str(BATCH_SIZE),
]
print(" ".join(embedding_command))
if RUN_FULL_EMBEDDINGS:
    subprocess.run(embedding_command, cwd=PROJECT_ROOT, check=True)
print("Embeddings exist:", EMBEDDINGS_PATH.exists())

## 5. Evaluate the lineage-held-out embedding baseline

Change `RUN_FULL_EMBEDDING_BASELINE` to `True` after embeddings exist.

In [ ]:
baseline_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "train_embedding_baseline.py"),
    "--embeddings",
    str(EMBEDDINGS_PATH),
    "--splits",
    str(SPLITS_PATH),
    "--output",
    str(METRICS_PATH),
]
print(" ".join(baseline_command))
if RUN_FULL_EMBEDDING_BASELINE:
    subprocess.run(baseline_command, cwd=PROJECT_ROOT, check=True)
if METRICS_PATH.exists():
    print(METRICS_PATH.read_text())

## 6. Next step

Compare this frozen-DNABERT2 result against the grouped-test k-mer reference. If the representation is promising, add repeated grouped splits and a known-QAC-gene baseline before increasing model complexity.